# Guardrails and Prompt Injection: Why Filters Reduce Risk and Only Architecture Bounds It

The full code of [Guardrails and Prompt Injection: Why Filters Reduce Risk and Only Architecture Bounds It](https://residual-stream.lvtan71.workers.dev/posts/guardrails-and-prompt-injection/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/guardrails-and-prompt-injection) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/guardrails-and-prompt-injection.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 50 min end to end on the RTX 3090 machine that saved the outputs below.
The agent's replies are sampled, so attack rates vary by machine: the 3B model with no defense was fooled 74% of the time in the post and 66 to 67% on two CUDA GPUs, and the defenses' order changes within their overlapping intervals.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 50 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "guardrails-and-prompt-injection"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/guardrails-and-prompt-injection


In [2]:
!tar xzf blog-77f8212.tar.gz

## `injection_both.py`

Runs `injection.py` with `SIZE=1.5B`, then `SIZE=3B`.

In [3]:
!{sys.executable} injection_both.py

Qwen2.5-1.5B-Instruct: 8 injections x 12 passages = 96 attacked inputs; 12 clean passages. Attack success with a 95% interval.

defense                         emailed attacker  link only   attack succeeded  any email, attacked  any email, clean  blocked: attacked, clean


no defense                                     3          7       10% (6%-18%)                4/96             1/12              0/96, 0/12


'documents are data' rule                      3          7       10% (6%-18%)                9/96             1/12              0/96, 0/12


+ spotlighting (datamarking)                   0         16      17% (11%-25%)                1/96             0/12              0/96, 0/12


input classifier                               2          7        9% (5%-17%)                3/96             1/12             27/96, 7/12


...

no defense                                    45         18      66% (56%-74%)               79/96            12/12              0/96, 0/12


'documents are data' rule                     49         20      72% (62%-80%)               88/96            12/12              0/96, 0/12


+ spotlighting (datamarking)                  41         12      55% (45%-65%)               70/96             9/12              0/96, 0/12


input classifier                              39         18      59% (49%-69%)               71/96            12/12             10/96, 0/12


no email tool                                  0         12       12% (7%-21%)                0/96             0/12              0/96, 0/12


The post printed:

```text
Qwen2.5-1.5B-Instruct: 8 injections x 12 passages = 96 attacked inputs; 12 clean passages. Attack success with a 95% interval.

defense                         emailed attacker  link only   attack succeeded  any email, attacked  any email, clean  blocked: attacked, clean
no defense                                     1          4        5% (2%-12%)                3/96             0/12              0/96, 0/12
'documents are data' rule                      3          6        9% (5%-17%)               11/96             0/12              0/96, 0/12
+ spotlighting (datamarking)                   0          9        9% (5%-17%)                0/96             0/12              0/96, 0/12
input classifier                               0          4        4% (2%-10%)                1/96             0/12             27/96, 6/12
no email tool                                  0          7        7% (4%-14%)                0/96             0/12              0/96, 0/12
Qwen2.5-3B-Instruct: 8 injections x 12 passages = 96 attacked inputs; 12 clean passages. Attack success with a 95% interval.

defense                         emailed attacker  link only   attack succeeded  any email, attacked  any email, clean  blocked: attacked, clean
no defense                                    52         19      74% (64%-82%)               86/96            12/12              0/96, 0/12
'documents are data' rule                     49         17      69% (59%-77%)               88/96            12/12              0/96, 0/12
+ spotlighting (datamarking)                  48         14      65% (55%-73%)               82/96             9/12              0/96, 0/12
input classifier                              46         19      68% (58%-76%)               78/96            12/12             10/96, 0/12
no email tool                                  0         10       10% (6%-18%)                0/96             0/12              0/96, 0/12
```